In [ ]:
import math

# ------------------------------------------------------------
# Utility Functions
# ------------------------------------------------------------

def is_prime(x):
    if x < 2:
        return False
    for i in range(2, int(math.sqrt(x)) + 1):
        if x % i == 0:
            return False
    return True


def next_power_of_two(x):
    p = 1
    while p < x:
        p *= 2
    return p


def prime_factors(x):
    factors = set()
    d = 2
    while d * d <= x:
        while x % d == 0:
            factors.add(d)
            x //= d
        d += 1
    if x > 1:
        factors.add(x)
    return list(factors)


def mod_inverse(a, mod):
    return pow(a, -1, mod)


# ------------------------------------------------------------
# Find Generator
# ------------------------------------------------------------

def find_generator(N):
    phi = N - 1
    factors = prime_factors(phi)
    for g in range(2, N):
        valid = True
        for p in factors:
            if pow(g, phi // p, N) == 1:
                valid = False
                break
        if valid:
            return g
    return None


# ------------------------------------------------------------
# DIF butterfly (forward NTT)
# Natural-order input, bit-reversed output
# Used internally to compute reference values
# ------------------------------------------------------------

def dif_ntt(a, omega, mod):
    n     = len(a)
    log2n = int(math.log2(n))
    mem   = a[:]
    tw    = [pow(omega, i, mod) for i in range(n)]
    for s in range(log2n):
        half     = n >> (s + 1)
        log_step = (log2n - 1) - s
        for p in range(n // 2):
            group = (p >> log_step) if half > 0 else p
            k     = (p & (half - 1)) if half > 0 else 0
            ii    = (group << (log2n - s)) | k
            jj    = ii | half
            tw_i  = k << s
            av, bv, twv = mem[ii], mem[jj], tw[tw_i]
            mem[ii] = (av + bv) % mod
            mem[jj] = (av - bv + mod) % mod * twv % mod
    return mem


# ------------------------------------------------------------
# DIT butterfly (inverse NTT)
# Bit-reversed input, natural-order output
# ------------------------------------------------------------

def dit_ntt(a, omega_inv, mod):
    n     = len(a)
    log2n = int(math.log2(n))
    mem   = a[:]
    tw    = [pow(omega_inv, i, mod) for i in range(n)]
    for s in range(log2n):
        half = 1 << s
        for p in range(n // 2):
            group = p >> s
            k     = p & (half - 1)
            ii    = (group << (s + 1)) | k
            jj    = ii | half
            step  = n >> (s + 1)
            tw_i  = k * step
            av, bv, twv = mem[ii], mem[jj], tw[tw_i]
            bw      = bv * twv % mod
            mem[ii] = (av + bw) % mod
            mem[jj] = (av - bw + mod) % mod
    return mem


# ------------------------------------------------------------
# Brute-force polynomial multiply (for verification only)
# ------------------------------------------------------------

def poly_mul(a, b):
    result = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            result[i + j] += x * y
    return result


# ------------------------------------------------------------
# Main NTT Parameter Generator
# ------------------------------------------------------------

def generate_ntt_parameters(a, b, output_file="ntt_params.txt"):

    # --------------------------------------------------------
    # Original lengths
    # --------------------------------------------------------
    len_a = len(a)
    len_b = len(b)

    # --------------------------------------------------------
    # Required convolution length
    # --------------------------------------------------------
    required_length = len_a + len_b - 1

    # --------------------------------------------------------
    # NTT length (next power of 2)
    # --------------------------------------------------------
    n = next_power_of_two(required_length)

    # --------------------------------------------------------
    # Zero padding
    # --------------------------------------------------------
    a = a + [0] * (n - len_a)
    b = b + [0] * (n - len_b)

    # --------------------------------------------------------
    # Estimate safe modulus lower bound
    # --------------------------------------------------------
    max_coeff_a = max(a)
    max_coeff_b = max(b)
    safe_limit  = max_coeff_a * max_coeff_b * min(len_a, len_b)
    minimum_N   = max(n + 1, safe_limit + 1)

    # --------------------------------------------------------
    # Find prime modulus N = k*n + 1
    # --------------------------------------------------------
    k = 1
    while True:
        N = k * n + 1
        if N >= minimum_N and is_prime(N):
            break
        k += 1

    # --------------------------------------------------------
    # Find generator g
    # --------------------------------------------------------
    g = find_generator(N)

    # --------------------------------------------------------
    # Primitive root:  omega = g^k mod N
    # --------------------------------------------------------
    omega = pow(g, k, N)

    # --------------------------------------------------------
    # Inverses
    # --------------------------------------------------------
    omega_inv = mod_inverse(omega, N)
    n_inv     = mod_inverse(n, N)

    # --------------------------------------------------------
    # Compute reference NTT values (for testbench verification)
    # --------------------------------------------------------
    a_ntt = dif_ntt(a[:], omega, N)
    b_ntt = dif_ntt(b[:], omega, N)
    c_ntt = [x * y % N for x, y in zip(a_ntt, b_ntt)]

    result_raw   = dit_ntt(c_ntt[:], omega_inv, N)
    result_scaled = [v * n_inv % N for v in result_raw]

    expected_full = poly_mul(a[:len_a], b[:len_b])
    expected_pad  = expected_full + [0] * (n - len(expected_full))

    # --------------------------------------------------------
    # Print to console
    # --------------------------------------------------------
    print("--------------------------------------------------")
    print("Padded A         =", a)
    print("Padded B         =", b)
    print("--------------------------------------------------")
    print("Transform Length =", n)
    print("Prime Modulus  N =", N)
    print("k                =", k)
    print("Generator g      =", g)
    print("Omega (w)        =", omega)
    print("Omega Inverse    =", omega_inv)
    print("n Inverse        =", n_inv)
    print("--------------------------------------------------")
    print("NTT(A)           =", a_ntt)
    print("NTT(B)           =", b_ntt)
    print("C = NTT(A)*NTT(B)=", c_ntt)
    print("INTT(C)          =", result_scaled)
    print("Expected         =", expected_pad)
    print("Match            =", result_scaled == expected_pad)
    print("--------------------------------------------------")

    # --------------------------------------------------------
    # Write ntt_params.txt
    # Format read by Verilog testbench:
    #   N_LEN     <value>
    #   MODULUS   <value>
    #   OMEGA     <value>
    #   OMEGA_INV <value>
    #   N_INV     <value>
    #   A <idx> <val>      (one line per coefficient, 0..n-1)
    #   B <idx> <val>
    #   ANTT <idx> <val>   (DIF NTT reference, for TEST 1 check)
    #   BNTT <idx> <val>   (DIF NTT reference, for TEST 2 check)
    #   CNTT <idx> <val>   (pointwise product reference)
    #   EXP  <idx> <val>   (expected final output, for TEST 3 check)
    # --------------------------------------------------------
    with open(output_file, "w") as f:
        f.write(f"N_LEN     {n}\n")
        f.write(f"MODULUS   {N}\n")
        f.write(f"OMEGA     {omega}\n")
        f.write(f"OMEGA_INV {omega_inv}\n")
        f.write(f"N_INV     {n_inv}\n")
        for i, v in enumerate(a):
            f.write(f"A {i} {v}\n")
        for i, v in enumerate(b):
            f.write(f"B {i} {v}\n")
        for i, v in enumerate(a_ntt):
            f.write(f"ANTT {i} {v}\n")
        for i, v in enumerate(b_ntt):
            f.write(f"BNTT {i} {v}\n")
        for i, v in enumerate(c_ntt):
            f.write(f"CNTT {i} {v}\n")
        for i, v in enumerate(expected_pad):
            f.write(f"EXP {i} {v}\n")

    print(f"\nntt_params.txt written to: {output_file}")


# ------------------------------------------------------------
# Input — change A and B here
# ------------------------------------------------------------

A = [1, 2, 3, 4]
B = [5, 4, 3, 2]

generate_ntt_parameters(A, B)